> ## SOLUTION / ANSWER KEY &mdash; Lab 10.4
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-04-classify-failure.ipynb`](../lab-04-classify-failure.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.4 &mdash; Classify the Failure Mode

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Map an observation symptom to a failure-mode label
- Cover wrong-tool, ungrounded, loop and format failures
- Turn debugging into symptom-to-fix pattern-matching

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Common failure modes & their fixes](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-10-04")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Debugging becomes a **field guide** once you can name the failure (deck slide 15): a trace **symptom**
maps to a **failure mode**, which maps to a **fix you already learned**. Wrong tool &rarr; better
descriptions; ungrounded &rarr; gather-first; loop &rarr; `max_iterations`; bad format &rarr; structured
output. Here you build the classifier.

In [ ]:
# Each symptom string comes from a trace observation or an error.
print("symptom -> failure mode -> known fix")

## Build it
Complete `classify`: map each symptom to its failure-mode label from a closed set.

In [ ]:
def classify(observation):
    o = observation.lower()
    if "unknown tool" in o:
        return "wrong_tool"
    if "hallucinat" in o or "ungrounded" in o:
        return "ungrounded_arg"
    if "max iterations" in o or "loop" in o:
        return "runaway_loop"
    if "could not parse" in o or "invalid json" in o:
        return "bad_format"
    return "unknown"

try:
    print(classify("unknown tool: lookup_order"))
    print(classify("argument was ungrounded"))
    print(classify("stopped: max iterations reached"))
    print(classify("could not parse output as JSON"))
    print(classify("everything looked fine"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- Each symptom lands on **one** label from a closed set &mdash; that's what turns debugging into pattern-matching.
- Every label points at a **fix from earlier in the course**: descriptions, gather-first, a step cap, structured output.
- `"unknown"` is the honest escape hatch &mdash; a symptom you can't yet name is itself a signal to read the trace more closely.

## Your turn (open task &mdash; no grader)
Add a new failure mode &mdash; e.g. map a `"rate limit"` symptom to `"provider_error"` (the exact thing
`with_backoff` handles). **What good looks like:** your classifier names it, and you can state the fix in one
line. A field guide is only useful if it covers the failures you actually hit.

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
def classify(observation):
    o = observation.lower()
    if "unknown tool" in o: return "wrong_tool"
    if "hallucinat" in o or "ungrounded" in o: return "ungrounded_arg"
    if "max iterations" in o or "loop" in o: return "runaway_loop"
    if "could not parse" in o or "invalid json" in o: return "bad_format"
    if "rate limit" in o or "429" in o: return "provider_error"   # fix: wrap the call in with_backoff
    return "unknown"
print(classify("Groq returned 429 rate limit exceeded"))   # -> provider_error

---
### Key takeaway
Name the failure and the fix falls out: wrong tool -> descriptions, ungrounded -> gather-first, loop -> max_iterations, bad format -> structured output. Every fix is a technique from this course; debugging is symptom-to-fix pattern-matching.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>